# EfficientNet-B0 Experimental Pipeline

This notebook evaluates EfficientNet-B0 on CIFAR-10 as part of the Green AI model compression experiments. The workflow follows the same structure as the ResNet-18 experiment: baseline training, pruning, quantization, evaluation, and comparison.

### Dataset Loading and Preprocessing

CIFAR-10 is loaded using torchvision and transformed into normalized tensors. Data loaders are created for batching the training and test data during model training and evaluation.

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import time
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


### Baseline EfficientNet-B0 Model

EfficientNet-B0 is initialized as the baseline architecture. The classifier layer is replaced so the model outputs ten classes for CIFAR-10 classification.

In [3]:
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b0
import time

# Initialise EfficientNet-B0 for CIFAR-10 classification
efficientnet_model = efficientnet_b0(weights=None)

# Replace classifier output layer for 10 CIFAR-10 classes
efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    10
)

efficientnet_model = efficientnet_model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientnet_model.parameters(), lr=0.001)

print("EfficientNet-B0 model initialized.")

EfficientNet-B0 model initialized.


### Baseline EfficientNet-B0 Training

The baseline model is trained for five epochs. Training loss, training accuracy, and total training time are recorded for comparison with the compressed models.

In [5]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Image preprocessing for CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Load CIFAR-10 training and test datasets
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Files already downloaded and verified
Files already downloaded and verified
Training samples: 50000
Testing samples: 10000
Training batches: 782
Testing batches: 157


In [8]:
num_epochs = 5

efficientnet_train_losses = []
efficientnet_train_accuracies = []

start_time = time.time()

for epoch in range(num_epochs):
    efficientnet_model.train()
    
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = efficientnet_model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct_predictions / total_samples
    
    efficientnet_train_losses.append(epoch_loss)
    efficientnet_train_accuracies.append(epoch_accuracy)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {epoch_loss:.4f} "
          f"Training Accuracy: {epoch_accuracy:.2f}%")

end_time = time.time()
efficientnet_training_time = end_time - start_time

print(f"\nTotal EfficientNet-B0 training time: {efficientnet_training_time:.2f} seconds")

Epoch [1/5] Loss: 0.8223 Training Accuracy: 71.59%
Epoch [2/5] Loss: 0.7794 Training Accuracy: 73.09%
Epoch [3/5] Loss: 0.7479 Training Accuracy: 74.34%
Epoch [4/5] Loss: 0.7895 Training Accuracy: 72.94%
Epoch [5/5] Loss: 0.7988 Training Accuracy: 72.72%

Total EfficientNet-B0 training time: 3020.31 seconds


### Baseline EfficientNet-B0 Evaluation

The trained baseline model is evaluated on the CIFAR-10 test set to measure predictive performance on unseen data.

In [9]:
efficientnet_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = efficientnet_model(images)
        
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

efficientnet_test_accuracy = 100 * correct / total

print(f"EfficientNet-B0 Test Accuracy: {efficientnet_test_accuracy:.2f}%")

EfficientNet-B0 Test Accuracy: 69.32%


### Baseline Model Checkpoint

The trained EfficientNet-B0 baseline model is saved so it can be reloaded for pruning and quantization experiments.

In [10]:
torch.save(efficientnet_model.state_dict(), "baseline_efficientnet_b0.pth")
print("Baseline EfficientNet-B0 model saved.")

Baseline EfficientNet-B0 model saved.


### Pruned EfficientNet-B0 Model

Unstructured L1 pruning is applied to convolutional and linear layers using a pruning amount of 20%. This allows the effect of weight sparsity on model performance to be evaluated.

In [11]:
import torch.nn.utils.prune as prune

# Apply unstructured pruning to convolutional and linear layers
efficientnet_pruning_amount = 0.20

for name, module in efficientnet_model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.l1_unstructured(module, name="weight", amount=efficientnet_pruning_amount)

print("Pruning applied to EfficientNet-B0 model.")

Pruning applied to EfficientNet-B0 model.


### Pruned Model Sparsity Check

The proportion of zero-valued parameters is calculated to confirm that the intended pruning level has been applied.

In [12]:
total_params = 0
zero_params = 0

for name, module in efficientnet_model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        weight = module.weight
        total_params += weight.nelement()
        zero_params += torch.sum(weight == 0).item()

efficientnet_sparsity = 100 * zero_params / total_params

print(f"EfficientNet-B0 sparsity after pruning: {efficientnet_sparsity:.2f}%")

EfficientNet-B0 sparsity after pruning: 20.00%


### Evaluation of Pruned EfficientNet-B0 Model

The pruned model is evaluated using the same test set as the baseline model. This allows the impact of pruning on predictive accuracy to be compared directly.

In [14]:
efficientnet_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = efficientnet_model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

efficientnet_pruned_test_accuracy = 100 * correct / total

print(f"Pruned EfficientNet-B0 Test Accuracy: {efficientnet_pruned_test_accuracy:.2f}%")

Pruned EfficientNet-B0 Test Accuracy: 68.71%


### Baseline vs Pruned EfficientNet-B0 Comparison

The performance of the baseline and pruned EfficientNet-B0 models is compared to evaluate the impact of pruning on predictive accuracy.

In [15]:
comparison_results = pd.DataFrame({
    "Model": ["Baseline EfficientNet-B0", "Pruned EfficientNet-B0"],
    "Test Accuracy (%)": [
        efficientnet_test_accuracy,
        efficientnet_pruned_test_accuracy
    ],
    "Sparsity (%)": [0, 20]
})

comparison_results

,Model,Test Accuracy (%),Sparsity (%)
0,Baseline EfficientNet-B0,69.32,0
1,Pruned EfficientNet-B0,68.71,20


In [16]:
comparison_results.to_csv(
    "efficientnet_baseline_vs_pruned.csv",
    index=False
)

print("EfficientNet comparison table saved.")

EfficientNet comparison table saved.


### Quantized EfficientNet-B0 Model

Dynamic quantization is applied to the EfficientNet-B0 model to reduce computational requirements while maintaining predictive performance.

In [17]:
import torch.quantization as quantization

quantized_efficientnet_model = torch.quantization.quantize_dynamic(
    efficientnet_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("EfficientNet-B0 quantization applied successfully.")

EfficientNet-B0 quantization applied successfully.


### Evaluation of Quantized EfficientNet-B0 Model

The quantized model is evaluated on the CIFAR-10 test set to assess whether dynamic quantization affects classification performance.

Dynamic post-training quantization is applied to a fresh copy of the trained baseline EfficientNet-B0 model to avoid interference from pruning masks.

In [19]:
from torchvision.models import efficientnet_b0
import torch.quantization
import torch.nn as nn

# Load a fresh EfficientNet-B0 model
quant_efficientnet_model = efficientnet_b0(weights=None)

# Replace classifier layer for CIFAR-10
quant_efficientnet_model.classifier[1] = nn.Linear(
    quant_efficientnet_model.classifier[1].in_features,
    10
)

# Load saved baseline weights
quant_efficientnet_model.load_state_dict(
    torch.load("baseline_efficientnet_b0.pth", map_location="cpu")
)

quant_efficientnet_model.eval()

# Apply dynamic post-training quantization
quantized_efficientnet_model = torch.quantization.quantize_dynamic(
    quant_efficientnet_model,
    {nn.Linear},
    dtype=torch.qint8
)

print("Fresh baseline EfficientNet-B0 loaded and quantization applied successfully.")

/tmp/ipykernel_14309/2161477293.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("baseline_efficientnet_b0.pth", map_location="cpu")


Fresh baseline EfficientNet-B0 loaded and quantization applied successfully.


In [20]:
quantized_efficientnet_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.cpu()
        labels = labels.cpu()

        outputs = quantized_efficientnet_model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

efficientnet_quantized_test_accuracy = 100 * correct / total

print(
    f"Quantized EfficientNet-B0 Test Accuracy: "
    f"{efficientnet_quantized_test_accuracy:.2f}%"
)

Quantized EfficientNet-B0 Test Accuracy: 69.30%


## Final EfficientNet-B0 Results

The baseline, pruned, and quantized EfficientNet-B0 models are compared to evaluate the impact of compression techniques on classification performance.

In [21]:
efficientnet_final_results = pd.DataFrame({
    "Model": [
        "Baseline EfficientNet-B0",
        "Pruned EfficientNet-B0",
        "Quantized EfficientNet-B0"
    ],
    "Test Accuracy (%)": [
        efficientnet_test_accuracy,
        efficientnet_pruned_test_accuracy,
        efficientnet_quantized_test_accuracy
    ],
    "Sparsity (%)": [
        0,
        20,
        0
    ],
    "Compression Method": [
        "None",
        "Pruning",
        "Dynamic Quantization"
    ]
})

efficientnet_final_results

,Model,Test Accuracy (%),Sparsity (%),Compression Method
0,Baseline EfficientNet-B0,69.32,0,None
1,Pruned EfficientNet-B0,68.71,20,Pruning
2,Quantized EfficientNet-B0,69.30,0,Dynamic Quantization


### Results Interpretation

The baseline EfficientNet-B0 model achieved a test accuracy of 69.32%.

Applying 20% unstructured pruning reduced accuracy slightly to 68.71%, indicating that moderate pruning can reduce model complexity with only a minor performance penalty.

Dynamic post-training quantization maintained nearly identical accuracy (69.30%) compared with the baseline model, suggesting that quantization can reduce computational requirements while preserving predictive performance.

Overall, dynamic quantization produced the best accuracy-compression trade-off for EfficientNet-B0 on the CIFAR-10 dataset.

In [22]:
efficientnet_final_results.to_csv(
    "efficientnet_final_results.csv",
    index=False
)

print("Final EfficientNet-B0 results saved.")

Final EfficientNet-B0 results saved.
